# Prompt, Context and Harness Engineering:

## Topics:

1. Zero shot prompting
2. One shot prompting
3. Few shot prompting
4. Chain of Thought (CoT)
5. Stepwise CoT
6. Self Consistency
7. Structured Outputs
   1. CoT with Strucutred Outputs
8. Speculative Decoding
9. Temperature
10. top_p
11. Harness Engineering

## Imports and Common Functions:

In [44]:
import re
import json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from pydantic import BaseModel, Field, EmailStr, field_validator
from dotenv import load_dotenv
from os import getenv
from enum import Enum
from typing import List, Optional

load_dotenv() # for any other environment variables you might have.
model = "openrouter/free"
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=getenv("OPENROUTER_API_KEY"),
)

In [2]:
def create_chat_message(role, message):
    return {
        "role": role,
        "content":message
    }
    
def create_user_message(message):
    return create_chat_message("user", message)

def create_system_message(message):
    return create_chat_message("system", message)

def create_assistant_message(message):
    return create_chat_message("assistant", message)

In [22]:
def SinglePrompt(messages):
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

## Zero Shot Prompting:

- We do not provide any sample input / output. We just ask the model a question and check it's response.
- Useful for very simple tasks where the output format etc does not matter.
- Limitations are accuracy, prompt sensitivity, Model biases.

In [5]:
system_msg = """
You are summarizer agent. You summarize whatever the context has been given and summarize it.
"""

prompt = """
The Industrial Revolution was a significant period of change from the 18th to 19th centuries, primarily affecting Europe and America. It marked a transition from agrarian, rural societies to industrial and urban ones. Before this era, manufacturing was typically done at home using simple tools. The revolution brought about powered machinery, factories, and mass production methods. Key industries like iron and textiles played crucial roles, as did the invention of the steam engine. This period also saw advancements in transportation, communication, and banking systems, fundamentally altering the economic and social landscape.
"""

print(SinglePrompt(
    [create_system_message(system_msg), create_user_message(prompt)]
))

**Summary:**

The Industrial Revolution (18th–19th centuries) was a transformative period that shifted Europe and America from agrarian, rural societies to industrial, urban ones. It replaced traditional home-based manufacturing with powered machinery, factories, and mass production, largely driven by innovations such as the steam engine and key industries like iron and textiles. Alongside these changes, advancements in transportation, communication, and banking reshaped the economic and social landscape.

**Key points:**
- **Time period:** 18th–19th centuries
- **Scope:** Primarily affected Europe and America
- **Change:** Transition from agrarian/rural to industrial/urban societies
- **Innovation:** Powered machinery, factories, mass production, and the steam engine
- **Impact:** Transformation of industries (iron, textiles) and broader economic/social systems


## One Shot Prompting:

- You provide an example input and output to tell the model of what kind of output is desired.
- Can specify required output format.
- Can specify the tone of the response - casual / professional etc.
- Useful for simpler tasks where one example is enough.

In [8]:
system_msg = """
Analyze the sentiment of the user's message.

Example input:
This tastes very good.

Example output:
<sentiment>Positive</sentiment>
<confidence>1.0</confidence>
"""

prompt = """
I am unsure about this cooking style. I did not enjoy it but there were other people who seemed to like it.
"""

print(SinglePrompt(
    [create_system_message(system_msg), create_user_message(prompt)]
))

<sentiment>Negative</sentiment>
<confidence>0.85</confidence>


## Few Shot Prompting:

- Useful for generating more accurate responses.
- Needed for complex tasks covering edge cases or special handling for certain use cases.
- Need to provide domain context or some information needed to answer the questions.
- This has a cost as more and more examples are provided the token usage increases hence costs increase.

In [9]:
system_msg = """
Parse chaotic, unstructured medical/clinical shorthand notes into a highly strict, normalized JSON schema.

Examples :-
---

Input: Pt presents with severe cephalalgia x3 days. Taking Advil 400mg PRN, gives mild relief. Denies emesis.
Output:
{
  "symptoms": [
    {"name": "Headache", "severity": "Severe", "duration": "3 days"}
  ],
  "medications": [
    {"brand_name": "Advil", "generic_name": "Ibuprofen", "dosage": "400mg", "frequency": "As needed", "efficacy": "Mild relief"}
  ],
  "negated_symptoms": ["Vomiting"]
}

---

Input: 54yo male c/o acute dyspnea and chest tightness since this morning. Ambulated to clinic. Hx of HTN.
Output:
{
  "symptoms": [
    {"name": "Shortness of breath", "severity": "Acute", "duration": "Since morning"},
    {"name": "Chest tightness", "severity": "Moderate", "duration": "Since morning"}
  ],
  "medications": [],
  "medical_history": ["Hypertension"],
  "negated_symptoms": []
}

"""

prompt = """
Kid has a barky cough and high fever (103F). Mom gave Tylenol liquid 5ml at 0200. No SOB noted by parent.
"""

print(SinglePrompt(
    [create_system_message(system_msg), create_user_message(prompt)]
))

{
  "symptoms": [
    {"name": "Barking cough", "severity": "Moderate", "duration": "Acute"},
    {"name": "Fever", "severity": "High (103°F)", "duration": "Acute"}
  ],
  "medications": [
    {"brand_name": "Tylenol", "generic_name": "Acetaminophen", "dosage": "5 mL", "frequency": "Single dose at 0200", "efficacy": "Unknown"}
  ],
  "medical_history": [],
  "negated_symptoms": ["Shortness of breath"]
}


## Multi step Prompting:

A usual way to get model to do something is to give a large prompt with examples and enough information so that it does not hallucinate.

---

The other way is to break down the prompt in individual steps. The complex task is broken down into a sequence of smaller tasks. This improves the model accuracy as we clearly specify what to focus on in each step and how to generate the next result as well.

In [10]:
system_msg = """
You are an expert business analyst.
Task: Analyze the provided product feedback. Work through these sequential steps:
• Step 1 (Extract): Identify and list the top 3 recurring customer complaints from the text.
• Step 2 (Translate): Rewrite those 3 complaints into plain, non-technical language that a customer service rep can understand.
• Step 3 (Format): Present the final output as a clean table with columns: Complaint, Root Cause Hypothesis, and Suggested Fix.
"""

prompt = """
Ever since the v4.2 update, the mobile app keeps throwing a 'Network Error 503' code whenever I try to navigate to the checkout page from a cellular network, though it seems to load fine on home Wi-Fi. Also, several users in the forums noted that the standard OAuth loop is broken—if they choose 'Sign in with Google', the pop-up window opens but immediately crashes back to the login screen without passing the authentication token. Finally, the UI rendering latency is unacceptable; the product images take up to 8 seconds to populate because the client-side app is trying to pull uncompressed 4K PNG assets directly from the main server rather than utilizing the CDN cached thumbnails.
"""

print(SinglePrompt(
    [create_system_message(system_msg), create_user_message(prompt)]
))

| Complaint | Root Cause Hypothesis | Suggested Fix |
|---|---|---|
| After the latest app update, the app sometimes shows a network error when customers try to check out using cellular data, though it works fine on home Wi-Fi. | The checkout backend or CDN may have a routing/configuration issue affecting mobile/cellular traffic, causing a 503 error. | Check backend logs for 503s on mobile; verify CDN and load balancer settings; add retry/fallback handling for checkout on cellular. |
| When customers try to sign in with Google, the pop-up opens and then immediately crashes/closes, so they cannot complete the login. | Likely an OAuth redirect URI or token-handling bug in the app. | Review Google OAuth client configuration and app deep-link handling; fix redirect URI and token exchange flow. |
| Product images take up to 8 seconds to load because the app pulls large, uncompressed images from the server instead of using cached thumbnails. | The app is not using CDN-cached or optimized ima

- Token usage increases but accuracy also increases.
- We specifically tell the LLM what to focus on.
- Such kind of prompting is very effective in reducing hallucinations.
- Biases do not usually come into picture here as the tasks are very small with clear defined goals so LLM focuses on facts.
- We can also divide this single prompt into multiple prompts where each task gets in the input the output of previous task. Through this we can also include HITL.

## Chain of Thought:

In [20]:
prompt = """
Roger has 5 tennis balls. He buys 2 cans of tennis balls. Each can has 3 tennis balls. How many balls does he have?
"""

print(SinglePrompt(
    [create_user_message(prompt)],
))

Roger starts with 5 tennis balls.

He buys 2 cans, each containing 3 tennis balls:
2 cans × 3 balls per can = 6 new tennis balls.

Total balls: 5 (original) + 6 (new) = **11 tennis balls**.


In [23]:
prompt = """
Roger has 5 tennis balls. He buys 2 cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now? Let's think step by step.
"""

print(SinglePrompt(
    [create_user_message(prompt)],
))

Let's work through this step by step!

**Step 1: Find how many tennis balls Roger already has.**
Roger starts with 5 tennis balls.

**Step 2: Figure out how many balls he bought.**
He buys 2 cans, and each can has 3 balls.
2 cans × 3 balls = 6 tennis balls

**Step 3: Add them together.**
5 (original balls) + 6 (new balls) = 11 tennis balls

**Roger now has 11 tennis balls!** 🎾


- In first prompt we did not tell the model to think step by step but it still got the correct result but with some weaker models or more complex tasks it would have failed.
- For second prompt we explicitly told to think step by step - for complex tasks this would have generated a more consistent and correct output.
  - the only difference in prompt is we tell LLM to think step by step.
    - In each iteration a token is generated, when we ask to figure it out step by step more tokens are used and more forward passes are used to answer the question.
    - Forcing intermediate steps lets the model write down sub-calculations (like math steps, logic premises, or code variables) before committing to a final conclusion.
    - Writing out words buys the model extra forward passes and processing space, which dramatically lowers the chance of skipping a logical leap on multistep problems like arithmetic or puzzles.
  - Newer models (like OpenAI's o-series or Claude's thinking modes) are built to reason by default. Adding the manual phrase to them often yields negligible gains, adds unnecessary latency and cost, or can occasionally mess up structured outputs like JSON
- We can also combine CoT with few shot prompting, where we actually also show the model how to think towards a problem in examples.

## Self Consistency:

- In this technique instead of just 1 CoT we create many CoTs and then do a majority voting.
  - It is an optimization technique on CoT.
  - We prompt the model multiple times and then choose the most common answer.
  - temperature is set to something like 0.7 for diverse output.
    - We can also give multiple prompts, however this makes harder to choose as answers would also depend on prompts.

In [39]:
prompt = """
Solve this problem step by step.

A server executes a batch of 100 sequential jobs, numbered from Job 1 to Job 100.
- Every 3rd job (Job 3, 6, 9, ...) requires a memory flush.
- Every 4th job (Job 4, 8, 12, ...) requires a disk checkpoint.
- HOWEVER, if a job number is a multiple of BOTH 3 and 4 (like Job 12), the system skips both the memory flush and the disk checkpoint, and instead runs a full reboot.
- Additionally, any job whose number ends in the digit '7' (Job 7, 17, 27, ...) is a diagnostic job. If a diagnostic job was scheduled for a memory flush or a disk checkpoint, that flush or checkpoint is cancelled (full reboots cannot be cancelled, though no multiple of 12 ends in 7 anyway).

Question: Across all 100 jobs, how many TOTAL memory flushes and disk checkpoints are actually executed (combined)?

Think step-by-step. At the very end of your response, output your final number inside tags like this: <ANSWER>number</ANSWER>.
"""

def extract_answer(text):
    match = re.search(r'<ANSWER>\s*(\d+)\s*</ANSWER>', text)
    if match:
        return int(match.group(1))
    numbers = re.findall(r'\b\d+\b', text)
    return int(numbers[-1]) if numbers else None

MODEL_NAME = "google/gemini-2.5-flash-lite"

# ==========================================
# 1. WITHOUT SAMPLES (Standard Greedy Output)
# ==========================================
print("=== APPROACH 1: STANDARD DECODING (NO SAMPLES) ===")
try:
    greedy_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    greedy_text = greedy_response.choices[0].message.content
    greedy_answer = extract_answer(greedy_text)

    print(f"Greedy Logic:\n{greedy_text}\n")
    print(f"➡️ Standard Single Output Answer: {greedy_answer}\n")
except Exception as e:
    print(f"Error in Greedy API Call: {e}")
    greedy_answer = None

# ==========================================
# 2. WITH SAMPLES (Self-Consistency)
# ==========================================
print("=== APPROACH 2: SELF-CONSISTENCY (7 SAMPLES) ===")
num_samples = 7
sc_answers = []

def fetch_single_sample(sample_idx):
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        # 0.5 - 0.6 is the sweet spot for modern instruction-tuned models:
        # enough diversity to try different mathematical setups without breaking arithmetic.
        temperature=0.55 
    )
    content = resp.choices[0].message.content
    return sample_idx, extract_answer(content)

try:
    with ThreadPoolExecutor(max_workers=num_samples) as executor:
        futures = [executor.submit(fetch_single_sample, i + 1) for i in range(num_samples)]
        
        for future in as_completed(futures):
            idx, ans = future.result()
            print(f"Path {idx} Answer: {ans}")
            if ans is not None:
                sc_answers.append(ans)

    if sc_answers:
        votes = Counter(sc_answers)
        majority_answer, count = votes.most_common(1)[0]

        print(f"\nAll Sampled Answers: {sc_answers}")
        print(f"🏆 Self-Consistency Majority Answer: {majority_answer} ({count}/{len(sc_answers)} votes)")
    else:
        print("Failed to extract any answers from the sampled responses.")
        majority_answer = None
        count = 0
except Exception as e:
    print(f"Error in SC API Call: {e}")
    majority_answer = None
    count = 0

# ==========================================
# COMPARISON SUMMARY
# ==========================================
print("\n=== SUMMARY OF CHANGE ===")
# Ground Truth Math Breakdown:
# 1. Multiples of 3 in 1..100: floor(100/3) = 33
# 2. Multiples of 4 in 1..100: floor(100/4) = 25
# 3. Multiples of 12 (both 3 and 4): floor(100/12) = 8
#    - Valid flushes (mult of 3, not 12): 33 - 8 = 25
#    - Valid checkpoints (mult of 4, not 12): 25 - 8 = 17
#    - Subtotal = 25 + 17 = 42
# 4. Cancelled by ending in '7':
#    - Multiples of 3 ending in 7 (within 1..100): 27, 57, 87 (3 jobs)
#    - Multiples of 4 ending in 7: None (even numbers never end in 7)
# 5. Final Executed Total: 42 - 3 = 39
correct_answer = 39

print(f"📌 Mathematical Ground Truth: {correct_answer}")
if greedy_answer == majority_answer:
    print(f"Both methods returned {majority_answer}.")
    if majority_answer == correct_answer:
        print(f"Both solved it accurately ({count}/{len(sc_answers)} votes).")
    else:
        print(f"Both methods fell for the same trap ({majority_answer} instead of {correct_answer}).")
else:
    print(f"🔄 The output CHANGED!")
    print(f"   Greedy Answer: {greedy_answer} {'(✅ Correct)' if greedy_answer == correct_answer else '(❌ Wrong)'}")
    print(f"   Self-Consistency Answer: {majority_answer} ({count}/{len(sc_answers)} votes) {'(✅ Correct)' if majority_answer == correct_answer else '(❌ Wrong)'}")

=== APPROACH 1: STANDARD DECODING (NO SAMPLES) ===
Greedy Logic:
Here's a step-by-step breakdown to solve this problem:

**1. Identify the base conditions for memory flushes and disk checkpoints:**

*   **Memory Flush:** Jobs that are multiples of 3.
*   **Disk Checkpoint:** Jobs that are multiples of 4.

**2. Identify the exception for combined multiples:**

*   Jobs that are multiples of both 3 and 4 (i.e., multiples of 12) trigger a full reboot instead of a flush or checkpoint.

**3. Identify the exception for diagnostic jobs:**

*   Jobs ending in '7' are diagnostic jobs.
*   If a diagnostic job was *scheduled* for a memory flush or disk checkpoint, that specific action is cancelled.

**4. Determine the range of jobs:**

*   The jobs are numbered from 1 to 100.

**5. Calculate the initial number of jobs scheduled for memory flushes:**

*   These are multiples of 3 up to 100.
*   Number of multiples of 3 = floor(100 / 3) = 33.

**6. Calculate the initial number of jobs scheduled for

```
=== APPROACH 1: STANDARD DECODING (NO SAMPLES) ===
Greedy Logic:
Here's a step-by-step breakdown to solve this problem:

**1. Identify the base conditions for memory flushes and disk checkpoints:**

*   **Memory Flush:** Jobs that are multiples of 3.
*   **Disk Checkpoint:** Jobs that are multiples of 4.

**2. Identify the exception for combined multiples:**

*   Jobs that are multiples of both 3 and 4 (i.e., multiples of 12) trigger a full reboot instead of a flush or checkpoint.

**3. Identify the exception for diagnostic jobs:**

*   Jobs ending in '7' are diagnostic jobs.
*   If a diagnostic job was *scheduled* for a memory flush or disk checkpoint, that specific action is cancelled.

**4. Determine the range of jobs:**

*   The jobs are numbered from 1 to 100.

**5. Calculate the initial number of jobs scheduled for memory flushes:**

*   These are multiples of 3 up to 100.
*   Number of multiples of 3 = floor(100 / 3) = 33.

**6. Calculate the initial number of jobs scheduled for disk checkpoints:**

*   These are multiples of 4 up to 100.
*   Number of multiples of 4 = floor(100 / 4) = 25.

**7. Calculate the number of jobs scheduled for a full reboot (multiples of 12):**

*   These are multiples of 12 up to 100.
*   Number of multiples of 12 = floor(100 / 12) = 8.
*   These 8 jobs *do not* get a memory flush or a disk checkpoint.

**8. Adjust the counts for memory flushes and disk checkpoints based on the reboot rule:**

*   The 8 jobs that are multiples of 12 were initially counted in both the multiples of 3 and multiples of 4.
*   So, we need to subtract these from the initial counts.
*   Jobs *initially* considered for memory flush (multiples of 3) = 33.
*   Jobs *initially* considered for disk checkpoint (multiples of 4) = 25.
*   Jobs that are multiples of 12 = 8.

*   Number of jobs *actually* scheduled for memory flush (excluding multiples of 12) = 33 - 8 = 25.
*   Number of jobs *actually* scheduled for disk checkpoint (excluding multiples of 12) = 25 - 8 = 17.

**9. Identify diagnostic jobs (ending in '7') within the range 1-100:**

*   Job 7
*   Job 17
*   Job 27
*   Job 37
*   Job 47
*   Job 57
*   Job 67
*   Job 77
*   Job 87
*   Job 97
*   There are 10 diagnostic jobs.

**10. Check if any diagnostic jobs were scheduled for memory flush or disk checkpoint:**

*   **Memory Flush:** Was any diagnostic job (ending in 7) a multiple of 3?
    *   7 (no)
    *   17 (no)
    *   27 (yes, 27 = 3 * 9)
    *   37 (no)
    *   47 (no)
    *   57 (yes, 57 = 3 * 19)
    *   67 (no)
    *   77 (no)
    *   87 (yes, 87 = 3 * 29)
    *   97 (no)
    *   So, Job 27, Job 57, and Job 87 were *scheduled* for a memory flush.

*   **Disk Checkpoint:** Was any diagnostic job (ending in 7) a multiple of 4?
    *   7 (no)
    *   17 (no)
    *   27 (no)
    *   37 (no)
    *   47 (no)
    *   57 (no)
    *   67 (no)
    *   77 (no)
    *   87 (no)
    *   97 (no)
    *   None of the diagnostic jobs are multiples of 4.

**11. Apply the diagnostic job cancellation rule:**

*   The diagnostic jobs that were scheduled for a memory flush are Job 27, Job 57, and Job 87. These flushes are cancelled.
*   We need to subtract these 3 cancelled flushes from our count of executed memory flushes.

**12. Calculate the final number of executed memory flushes:**

*   Number of jobs *actually* scheduled for memory flush (excluding multiples of 12) = 25.
*   Number of memory flushes cancelled due to diagnostic jobs = 3.
*   Total executed memory flushes = 25 - 3 = 22.

**13. Calculate the final number of executed disk checkpoints:**

*   Number of jobs *actually* scheduled for disk checkpoint (excluding multiples of 12) = 17.
*   Number of disk checkpoints cancelled due to diagnostic jobs = 0 (as no diagnostic jobs were multiples of 4).
*   Total executed disk checkpoints = 17 - 0 = 17.

**14. Calculate the TOTAL memory flushes and disk checkpoints executed:**

*   Total executed = Executed memory flushes + Executed disk checkpoints
*   Total executed = 22 + 17 = 39.

<ANSWER>39</ANSWER>

➡️ Standard Single Output Answer: 39

=== APPROACH 2: SELF-CONSISTENCY (7 SAMPLES) ===
Path 2 Answer: 39
Path 1 Answer: 39
Path 3 Answer: 39
Path 7 Answer: 39
Path 5 Answer: 39
Path 4 Answer: 39
Path 6 Answer: 39

All Sampled Answers: [39, 39, 39, 39, 39, 39, 39]
🏆 Self-Consistency Majority Answer: 39 (7/7 votes)

=== SUMMARY OF CHANGE ===
📌 Mathematical Ground Truth: 39
Both methods returned 39.
Both solved it accurately (7/7 votes).
```

- In this case both greedy and majority voting gave the same result. If we used a weaker model probably the results would have been different.

In [40]:
# Upgraded Prompt: A 10-day state machine with two interacting rules on overlapping days (Day 6).
# Greedy decoding (Temp 0.0) frequently miscalculates the Day 6 compound interaction or lookback window.
prompt = """
Let's solve a tricky inventory simulation problem step by step.

A warehouse tracks crates over 10 days (Day 1 to Day 10) following these exact rules in daily order:

1. Morning Delivery (happens FIRST each day):
   - Day 1: 2 crates delivered.
   - Day 2: 3 crates delivered.
   - Day 3 and onwards: Today's delivery is the sum of the deliveries from the previous TWO days, minus 2.

2. Afternoon Inspection (happens SECOND, after Morning Delivery is added to the total):
   - On every 3rd day (Day 3, Day 6, Day 9), an inspector removes 1/3 of the current total crates in the warehouse (rounded DOWN to the nearest whole crate).

3. Evening Audit (happens THIRD, at the very end of the day):
   - On every even-numbered day (Day 2, Day 4, Day 6, Day 8, Day 10), if the current total of crates in the warehouse is strictly greater than 10, a tax of 3 crates is removed. Otherwise, 1 bonus crate is added.

Question: Exactly how many total crates are in the warehouse at the very end of Day 10 (after the Evening Audit)?

Think step-by-step for each day from Day 1 to Day 10, explicitly writing down:
- Morning Delivery amount (and new running total)
- Afternoon Inspection removal, if applicable (and new running total)
- Evening Audit adjustment, if applicable (and final end-of-day total)

At the very end of your response, output your final number inside tags like this: <ANSWER>number</ANSWER>.
"""

def extract_answer(text):
    # First, try to find the explicit tagged answer
    match = re.search(r'<ANSWER>\s*(\d+)\s*</ANSWER>', text)
    if match:
        return int(match.group(1))
    
    # Fallback: find the last number in the text
    numbers = re.findall(r'\b\d+\b', text)
    return int(numbers[-1]) if numbers else None

MODEL_NAME = "google/gemini-2.5-flash-lite"

# ==========================================
# 1. WITHOUT SAMPLES (Standard Greedy Output)
# ==========================================
print("=== APPROACH 1: STANDARD DECODING (NO SAMPLES) ===")
try:
    greedy_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0 # Forces deterministic, greedy path
    )
    greedy_text = greedy_response.choices[0].message.content
    greedy_answer = extract_answer(greedy_text)

    print(f"Greedy Logic:\n{greedy_text}\n")
    print(f"➡️ Standard Single Output Answer: {greedy_answer}\n")
except Exception as e:
    print(f"Error in Greedy API Call: {e}")
    greedy_answer = None

# ==========================================
# 2. WITH SAMPLES (Self-Consistency)
# ==========================================
print("=== APPROACH 2: SELF-CONSISTENCY (7 SAMPLES) ===")
num_samples = 7
sc_answers = []

def fetch_single_sample(sample_idx):
    """Fires a single independent API call with high temperature."""
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8 # High enough to explore different reasoning paths
    )
    content = resp.choices[0].message.content
    return sample_idx, extract_answer(content)

try:
    # Fire all 7 requests in parallel because OpenRouter ignores `n=7`
    with ThreadPoolExecutor(max_workers=num_samples) as executor:
        futures = [executor.submit(fetch_single_sample, i + 1) for i in range(num_samples)]
        
        for future in as_completed(futures):
            idx, ans = future.result()
            print(f"Path {idx} Answer: {ans}")
            if ans is not None:
                sc_answers.append(ans)

    if sc_answers:
        votes = Counter(sc_answers)
        majority_answer, count = votes.most_common(1)[0]

        print(f"\nAll Sampled Answers: {sc_answers}")
        print(f"🏆 Self-Consistency Majority Answer: {majority_answer} ({count}/{len(sc_answers)} votes)")
    else:
        print("Failed to extract any answers from the sampled responses.")
        majority_answer = None
        count = 0
except Exception as e:
    print(f"Error in SC API Call: {e}")
    majority_answer = None
    count = 0

# ==========================================
# COMPARISON SUMMARY
# ==========================================
print("\n=== SUMMARY OF CHANGE ===")
# Ground Truth Math Breakdown:
# Deliveries (D_n = D_{n-1} + D_{n-2} - 2):
# D1=2, D2=3, D3=3, D4=4, D5=5, D6=7, D7=10, D8=15, D9=23, D10=36
#
# Daily Totals (Start -> +Delivery -> -Inspection(Day 3,6,9) -> Audit(Even Days)):
# Day 1: 0 + 2 = 2
# Day 2: 2 + 3 = 5 -> Audit (<=10, +1) = 6
# Day 3: 6 + 3 = 9 -> Inspect (-floor(9/3)=3) = 6
# Day 4: 6 + 4 = 10 -> Audit (<=10, +1) = 11
# Day 5: 11 + 5 = 16
# Day 6: 16 + 7 = 23 -> Inspect (-floor(23/3)=7) = 16 -> Audit (>10, -3) = 13
# Day 7: 13 + 10 = 23
# Day 8: 23 + 15 = 38 -> Audit (>10, -3) = 35
# Day 9: 35 + 23 = 58 -> Inspect (-floor(58/3)=19) = 39
# Day 10: 39 + 36 = 75 -> Audit (>10, -3) = 72
correct_answer = 72

if greedy_answer == majority_answer:
    print(f"Both methods returned {majority_answer}.")
    if majority_answer == correct_answer:
        print(f"The model solved it greedily. Self-Consistency validated it ({count}/{len(sc_answers)} votes).")
    else:
        print(f"Both methods failed. The correct answer is {correct_answer}.")
else:
    print(f"🔄 The output CHANGED!")
    print(f"   Without samples, the model got stuck on a reasoning error and guessed: {greedy_answer}")
    print(f"   With self-consistency, the majority of paths voted for: {majority_answer} ({count}/{len(sc_answers)} votes)")
    
    if majority_answer == correct_answer:
        print("\n🎉 Self-Consistency successfully fixed the hallucination of the Greedy approach!")

=== APPROACH 1: STANDARD DECODING (NO SAMPLES) ===
Greedy Logic:
Let's track the inventory day by day.

**Day 1:**

*   **Morning Delivery:** 2 crates delivered.
    *   Running total: 0 + 2 = 2 crates.
*   **Afternoon Inspection:** Not applicable (Day 1 is not a 3rd day).
    *   Running total: 2 crates.
*   **Evening Audit:** Day 1 is odd. The current total (2) is not strictly greater than 10. So, 1 bonus crate is added.
    *   End-of-day total: 2 + 1 = 3 crates.

**Day 2:**

*   **Morning Delivery:** 3 crates delivered.
    *   Running total: 3 (from end of Day 1) + 3 = 6 crates.
*   **Afternoon Inspection:** Not applicable (Day 2 is not a 3rd day).
    *   Running total: 6 crates.
*   **Evening Audit:** Day 2 is even. The current total (6) is not strictly greater than 10. So, 1 bonus crate is added.
    *   End-of-day total: 6 + 1 = 7 crates.

**Day 3:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 1: 2, Day 2: 3) minus 2. So, 2 + 3 - 2 = 3 crates delivered.
    

```
=== APPROACH 1: STANDARD DECODING (NO SAMPLES) ===
Greedy Logic:
Let's track the inventory day by day.

**Day 1:**

*   **Morning Delivery:** 2 crates delivered.
    *   Running total: 0 + 2 = 2 crates.
*   **Afternoon Inspection:** Not applicable (Day 1 is not a 3rd day).
    *   Running total: 2 crates.
*   **Evening Audit:** Day 1 is odd. The current total (2) is not strictly greater than 10. So, 1 bonus crate is added.
    *   End-of-day total: 2 + 1 = 3 crates.

**Day 2:**

*   **Morning Delivery:** 3 crates delivered.
    *   Running total: 3 (from end of Day 1) + 3 = 6 crates.
*   **Afternoon Inspection:** Not applicable (Day 2 is not a 3rd day).
    *   Running total: 6 crates.
*   **Evening Audit:** Day 2 is even. The current total (6) is not strictly greater than 10. So, 1 bonus crate is added.
    *   End-of-day total: 6 + 1 = 7 crates.

**Day 3:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 1: 2, Day 2: 3) minus 2. So, 2 + 3 - 2 = 3 crates delivered.
    *   Running total: 7 (from end of Day 2) + 3 = 10 crates.
*   **Afternoon Inspection:** Day 3 is a 3rd day. Inspector removes 1/3 of the current total (10). 10 / 3 = 3.33. Rounded down is 3 crates removed.
    *   Running total: 10 - 3 = 7 crates.
*   **Evening Audit:** Day 3 is odd. Not applicable.
    *   End-of-day total: 7 crates.

**Day 4:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 2: 3, Day 3: 3) minus 2. So, 3 + 3 - 2 = 4 crates delivered.
    *   Running total: 7 (from end of Day 3) + 4 = 11 crates.
*   **Afternoon Inspection:** Not applicable (Day 4 is not a 3rd day).
    *   Running total: 11 crates.
*   **Evening Audit:** Day 4 is even. The current total (11) is strictly greater than 10. So, 3 crates are removed.
    *   End-of-day total: 11 - 3 = 8 crates.

**Day 5:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 3: 3, Day 4: 4) minus 2. So, 3 + 4 - 2 = 5 crates delivered.
    *   Running total: 8 (from end of Day 4) + 5 = 13 crates.
*   **Afternoon Inspection:** Not applicable (Day 5 is not a 3rd day).
    *   Running total: 13 crates.
*   **Evening Audit:** Day 5 is odd. Not applicable.
    *   End-of-day total: 13 crates.

**Day 6:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 4: 4, Day 5: 5) minus 2. So, 4 + 5 - 2 = 7 crates delivered.
    *   Running total: 13 (from end of Day 5) + 7 = 20 crates.
*   **Afternoon Inspection:** Day 6 is a 3rd day. Inspector removes 1/3 of the current total (20). 20 / 3 = 6.66. Rounded down is 6 crates removed.
    *   Running total: 20 - 6 = 14 crates.
*   **Evening Audit:** Day 6 is even. The current total (14) is strictly greater than 10. So, 3 crates are removed.
    *   End-of-day total: 14 - 3 = 11 crates.

**Day 7:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 5: 5, Day 6: 7) minus 2. So, 5 + 7 - 2 = 10 crates delivered.
    *   Running total: 11 (from end of Day 6) + 10 = 21 crates.
*   **Afternoon Inspection:** Not applicable (Day 7 is not a 3rd day).
    *   Running total: 21 crates.
*   **Evening Audit:** Day 7 is odd. Not applicable.
    *   End-of-day total: 21 crates.

**Day 8:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 6: 7, Day 7: 10) minus 2. So, 7 + 10 - 2 = 15 crates delivered.
    *   Running total: 21 (from end of Day 7) + 15 = 36 crates.
*   **Afternoon Inspection:** Not applicable (Day 8 is not a 3rd day).
    *   Running total: 36 crates.
*   **Evening Audit:** Day 8 is even. The current total (36) is strictly greater than 10. So, 3 crates are removed.
    *   End-of-day total: 36 - 3 = 33 crates.

**Day 9:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 7: 10, Day 8: 15) minus 2. So, 10 + 15 - 2 = 23 crates delivered.
    *   Running total: 33 (from end of Day 8) + 23 = 56 crates.
*   **Afternoon Inspection:** Day 9 is a 3rd day. Inspector removes 1/3 of the current total (56). 56 / 3 = 18.66. Rounded down is 18 crates removed.
    *   Running total: 56 - 18 = 38 crates.
*   **Evening Audit:** Day 9 is odd. Not applicable.
    *   End-of-day total: 38 crates.

**Day 10:**

*   **Morning Delivery:** Sum of previous two deliveries (Day 8: 15, Day 9: 23) minus 2. So, 15 + 23 - 2 = 36 crates delivered.
    *   Running total: 38 (from end of Day 9) + 36 = 74 crates.
*   **Afternoon Inspection:** Not applicable (Day 10 is not a 3rd day).
    *   Running total: 74 crates.
*   **Evening Audit:** Day 10 is even. The current total (74) is strictly greater than 10. So, 3 crates are removed.
    *   End-of-day total: 74 - 3 = 71 crates.

<ANSWER>71</ANSWER>

➡️ Standard Single Output Answer: 71

=== APPROACH 2: SELF-CONSISTENCY (7 SAMPLES) ===
Path 4 Answer: 73
Path 3 Answer: 72
Path 1 Answer: 71
Path 6 Answer: 72
Path 7 Answer: 71
Path 5 Answer: 69
Path 2 Answer: 72

All Sampled Answers: [73, 72, 71, 72, 71, 69, 72]
🏆 Self-Consistency Majority Answer: 72 (3/7 votes)

=== SUMMARY OF CHANGE ===
🔄 The output CHANGED!
   Without samples, the model got stuck on a reasoning error and guessed: 71
   With self-consistency, the majority of paths voted for: 72 (3/7 votes)

🎉 Self-Consistency successfully fixed the hallucination of the Greedy approach!
```

- In this prompt we can see that the greedy strategy if choosing the best did not work out as the no sampling one guessed the anser as 71 where as self consistency fixed it via majority voting.

## Structured Output:

In [43]:
system_msg = """
You are an automated customer support triage agent. 
Extract key information from the customer's email and categorize the issue strictly following the provided JSON schema.
"""

# A sample customer email
prompt = """
From: John Doe <j.doe@example.com>
Message: Hi, I ordered two 'SuperWidget 3000s' on Tuesday, but only received one. 
My order number is #ORD-99321. Also, the one I did get has a huge scratch on the screen. 
I want a refund for the missing one and a replacement for the scratched one. 
Please fix this immediately, I am extremely frustrated!
"""

# Define the exact JSON schema you want the model to output
ticket_triage_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "support_ticket_extraction",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "customer_name": {
                    "type": "string",
                    "description": "The name of the customer."
                },
                "customer_email": {
                    "type": "string",
                    "description": "The email address of the customer."
                },
                "order_number": {
                    "type": ["string", "null"],
                    "description": "The order number if mentioned, otherwise null."
                },
                "issue_categories": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["refund_request", "replacement_request", "missing_item", "damaged_item", "technical_support", "other"]
                    },
                    "description": "Categorize the issues mentioned in the email."
                },
                "products_mentioned": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of product names mentioned."
                },
                "sentiment": {
                    "type": "string",
                    "enum": ["positive", "neutral", "frustrated", "angry"],
                    "description": "The emotional tone of the email."
                },
                "requires_urgent_escalation": {
                    "type": "boolean",
                    "description": "True if the customer is angry or demands immediate action."
                }
            },
            "required": [
                "customer_name", 
                "customer_email", 
                "order_number", 
                "issue_categories", 
                "products_mentioned", 
                "sentiment", 
                "requires_urgent_escalation"
            ],
            "additionalProperties": False
        }
    }
}

response = client.chat.completions.create(
    model=model,
    messages=[create_system_message(system_msg), create_user_message(prompt)],
    response_format=ticket_triage_schema
)

# Parse the JSON string returned by the model
result_json = json.loads(response.choices[0].message.content)
print(json.dumps(result_json, indent=2))

{
  "customer_email": "j.doe@example.com",
  "customer_name": "John Doe",
  "issue_categories": [
    "missing_item",
    "damaged_item"
  ],
  "order_number": "#ORD-99321",
  "products_mentioned": [
    "SuperWidget 3000"
  ],
  "requires_urgent_escalation": true,
  "sentiment": "angry"
}


```
{
  "customer_email": "j.doe@example.com",
  "customer_name": "John Doe",
  "issue_categories": [
    "missing_item",
    "damaged_item"
  ],
  "order_number": "#ORD-99321",
  "products_mentioned": [
    "SuperWidget 3000"
  ],
  "requires_urgent_escalation": true,
  "sentiment": "angry"
}
```

- We parsed the output of LLM by loading it into a json. `result_json = json.loads(response.choices[0].message.content)`
- The schema we defined is very verbose and it works, it's just a lot of work to do to define schema.

In [46]:
# 1. Define strict choices using Python Enums
class IssueCategory(str, Enum):
    refund_request = "refund_request"
    replacement_request = "replacement_request"
    missing_item = "missing_item"
    damaged_item = "damaged_item"
    technical_support = "technical_support"
    other = "other"

class Sentiment(str, Enum):
    positive = "positive"
    neutral = "neutral"
    frustrated = "frustrated"
    angry = "angry"

# 2. Define the exact output structure using Pydantic
class SupportTicketExtraction(BaseModel):
    customer_name: str = Field(description="The name of the customer.")
    customer_email: str = Field(description="The email address of the customer.")
    order_number: Optional[str] = Field(
        description="The order number if mentioned, otherwise null."
    )
    issue_categories: List[IssueCategory] = Field(
        description="Categorize the issues mentioned in the email."
    )
    products_mentioned: List[str] = Field(
        description="List of product names mentioned."
    )
    sentiment: Sentiment = Field(description="The emotional tone of the email.")
    requires_urgent_escalation: bool = Field(
        description="True if the customer is angry or demands immediate action."
    )

# 3. Define the prompts
system_msg = """
You are an automated customer support triage agent. 
Extract key information from the customer's email and categorize the issue.
"""

prompt = """
From: John Doe <j.doe@example.com>
Message: Hi, I ordered two 'SuperWidget 3000s' on Tuesday, but only received one. 
My order number is #ORD-99321. Also, the one I did get has a huge scratch on the screen. 
I want a refund for the missing one and a replacement for the scratched one. 
Please fix this immediately, I am extremely frustrated!
"""

# 4. Call the API using .parse() and pass the Pydantic model
response = client.chat.completions.parse(
    model=model,
    messages=[create_system_message(system_msg), create_user_message(prompt)],
    response_format=SupportTicketExtraction
)

# 5. Access the parsed Python object directly (fully typed)
ticket = response.choices[0].message.parsed

print(f"Customer Name: {ticket.customer_name}")
print(f"Customer Email: {ticket.customer_email}")
print(f"Order Number: {ticket.order_number}")
print(f"Sentiment: {ticket.sentiment.value}")
print(f"Requires Escalation: {ticket.requires_urgent_escalation}")

print("\nIssues:")
for issue in ticket.issue_categories:
    print(f" - {issue.value}")

print("\nProducts:")
for product in ticket.products_mentioned:
    print(f" - {product}")

Customer Name: John Doe
Customer Email: j.doe@example.com
Order Number: #ORD-99321
Sentiment: frustrated
Requires Escalation: True

Issues:
 - missing_item
 - damaged_item
 - refund_request
 - replacement_request

Products:
 - SuperWidget 3000s


```
Customer Name: John Doe
Customer Email: j.doe@example.com
Order Number: #ORD-99321
Sentiment: frustrated
Requires Escalation: False

Issues:
 - missing_item
 - damaged_item

Products:
 - SuperWidget 3000s
```

- Here we got a object instead of dictionary.
- this is a lot easier to create / maintain / debug and is more readable than the large schema format in previous example.
- notice how we use the `parsed` here instead of the normal `content`. `ticket = response.choices[0].message.parsed`
  - Also using the parse method of chat completions API. `client.chat.completions.parse`

#### Structured Output with CoT

In [48]:
# 1. Define the schema with a reasoning field FIRST
class MathSolution(BaseModel):
    step_by_step_reasoning: List[str] = Field(
        description="Break down the math problem step by step before answering."
    )
    final_answer: int = Field(
        description="The final calculated number."
    )

system_msg = "You are a logical math solver. Always think step-by-step."
prompt = "I have 5 apples. I buy 7 more. I give half of all my apples to a friend, and then I eat 2. How many do I have left?"

# 2. Call the API
response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ],
    response_format=MathSolution
)

solution = response.choices[0].message.parsed

# 3. Print the results
print("--- AI's Thought Process ---")
for i, step in enumerate(solution.step_by_step_reasoning, 1):
    print(f"Step {i}: {step}")

print(f"\nFinal Answer: {solution.final_answer}")

--- AI's Thought Process ---
Step 1: Start with the initial amount of apples: 5 apples.
Step 2: Buy additional apples: 5 apples + 7 apples = 12 apples.
Step 3: Give half of the total apples to a friend: 12 apples / 2 = 6 apples given away, so I have 12 apples - 6 apples = 6 apples left.
Step 4: Eat 2 apples: 6 apples - 2 apples = 4 apples left.

Final Answer: 4


```
--- AI's Thought Process ---
Step 1: Start with the initial amount of apples: 5 apples.
Step 2: Buy additional apples: 5 apples + 7 apples = 12 apples.
Step 3: Give half of the total apples to a friend: 12 apples / 2 = 6 apples given away, so I have 12 apples - 6 apples = 6 apples left.
Step 4: Eat 2 apples: 6 apples - 2 apples = 4 apples left.

Final Answer: 4
```

- When using `client.chat.completions.parse()`, the OpenAI SDK converts your Pydantic class into a `json_schema` payload under the hood. However, strict JSON Schema enforcement `(response_format={"type": "json_schema", ...})` is only supported by specific models and providers:
Using OpenRouter / a non-OpenAI provider: Many open-weight models or free-tier endpoints on OpenRouter (like certain Llama, Mistral, Nemotron, or Cohere endpoints) silently ignore the response_format parameter if the underlying hosting provider doesn't support constrained decoding.
   - Hence we explicitly set the model to `gpt-4o-mini` here

## OpenAI now recommends using the Response API instead of Chat Completions API :-

- Migration guide :- https://developers.openai.com/api/docs/guides/migrate-to-responses?lang=python
- chat messages types remain the same.
- for textual input the prompt etc can remain the same.
- It offers better pricing as caches are used here.

## Speculative Decoding:

**Speculative decoding** is a clever trick to make Large Language Models (LLMs) generate text 2x to 3x faster, without changing the final output or losing any quality.

### The Problem
LLMs are slow because they are **memory-bound**. Generating a single word requires loading the entire model's massive weights into the GPU's memory. The GPU spends most of its time waiting for data to load, not doing actual math. 

However, because the GPU has massive parallel computing power, *checking* multiple words at once takes almost the exact same amount of time as checking just one.

### The Solution: Draft and Verify
Speculative decoding takes advantage of this unused parallel power by teaming up two models:

1. **The Drafter (Small & Fast):** A tiny, lightweight model quickly guesses the next few words (e.g., 4 tokens) one by one. 
2. **The Target (Large & Accurate):** The main, massive model looks at all 4 guessed words and verifies them **all at once in parallel** (in a single step).
3. **Accept or Fix:** 
   - If the small model guessed correctly, the big model accepts them. You just generated 4 words in the time it normally takes to generate 1!
   - If the small model makes a mistake (say, on the 3rd word), the big model accepts the first two, fixes the 3rd, and throws away the 4th.

### The Bottom Line
It is **mathematically lossless**. Because the large target model always has the final say and corrects any mistakes, the final text is exactly identical to what the large model would have generated on its own - it just gets there much faster.

### Limitations of Speculative Decoding

While it speeds up generation, speculative decoding has a few key drawbacks:

* **Requires a Matching Draft Model:** You need a small model that thinks very similarly to your large target model. If the small model guesses poorly, the large model rejects the tokens, and the extra drafting steps actually make the overall process slower.
* **Increased Memory (VRAM) Usage:** You must load both the large target model and the small draft model into the GPU's memory simultaneously, which can be an issue if you are already tight on VRAM.
* **Fails at High Concurrency:** Speculative decoding only helps when the system is processing one or a few requests at a time (memory-bound). If you are processing large batches of users at once, the GPU is already fully utilizing its compute power, and the drafting process just adds unnecessary overhead.
* **Struggles with Complex Tasks:** For highly nuanced tasks like advanced coding, logic puzzles, or niche languages, the small draft model's accuracy plummets. This leads to a low "acceptance rate" and eliminates the speed benefit.

## Sampling using Temperature:

**Temperature** is a setting that controls how creative, random, or predictable a Large Language Model's (LLM) responses will be.

### The Concept
When an LLM generates text, it doesn't just pick one word; it assigns a probability score to *all* possible next words. Temperature is a mathematical dial that adjusts these scores before the model makes its final choice.

### How It Works
*   **Low Temperature (e.g., 0.1 to 0.4):** Sharpens the choices. The model plays it safe and heavily favors the most likely next word. The output is highly focused, predictable, and factual. 
    * *Best for:* Coding, math, data extraction, and strict factual Q&A.
*   **High Temperature (e.g., 0.7 to 1.0+):** Flattens the choices. Less likely words get a boost, giving them a fair chance to be chosen. The output is diverse, surprising, and creative. 
    * *Best for:* Brainstorming, poetry, marketing copy, and storytelling.
*   **Temperature = 0:** Known as "greedy decoding." The model simply chooses the #1 most probable word every single time, with zero randomness.

### The Bottom Line
Think of temperature as a "creativity vs. precision" dial. Lower it when you need exact answers, and raise it when you want variety and imagination.

### Limitations of Temperature

*   **Too High = Gibberish:** If the temperature is dialed up too much (usually above 1.2 - 1.5), the model starts picking completely random, highly unlikely words. This leads to chaotic, nonsensical text and severe hallucinations.
*   **Too Low = Robotic and Repetitive:** If you use a low temperature for a creative task, the text becomes dry and boring. Worse, the model can easily get stuck repeating the exact same phrase over and over in an endless loop.
*   **It's Just a Filter, Not a Brain:** Changing the temperature doesn't change the model's underlying knowledge. It won't make a model smarter or fix a vague prompt; it only changes how it rolls the dice when picking vocabulary.

## Top-p (Nucleus Sampling): A Quick Overview

**Top-p** (also known as nucleus sampling) is a dynamic filter that controls how many vocabulary options a Large Language Model considers before making a choice, based on their combined probability.

### The Concept
Instead of considering the entire dictionary for the next word, the model sorts all possible next words from most likely to least likely. It then goes down the list, adding up their probability scores, and stops when it hits a target percentage (the "p" value). Everything below that cutoff is immediately discarded.

### How It Works
*   **Low Top-p (e.g., 0.1 to 0.5):** A strict filter. If you set top-p to 0.2, the model only considers the very top words whose probabilities add up to 20%. The pool of choices is tiny, making the output highly focused, safe, and on-topic.
*   **High Top-p (e.g., 0.8 to 1.0):** A loose filter. Setting top-p to 0.9 means the model considers a wide pool of words making up 90% of the probability mass. This allows less obvious words to slip through, resulting in more varied and expressive text.
*   **Top-p = 1.0:** The filter is effectively off. The model considers every single word in its vocabulary.

### The Bottom Line
While *temperature* changes the probability scores themselves, *top-p* chops off the long tail of low-probability words. Think of it as a bouncer at a club: a low top-p only lets the VIPs (highly likely words) in, while a high top-p lets almost everyone in.

### Limitations of Top-p

*   **Clashes with Temperature:** Because temperature and top-p both control randomness, tweaking both at the exact same time can lead to unpredictable, hard-to-tune results. The general industry best practice is to **change one and leave the other at its default** (e.g., adjust temperature, keep top-p at 1.0).
*   **Not a Guarantee of Quality:** If the model is confused by a bad prompt, the "top 90%" of words might all be poor choices. Top-p just forces the model to pick from that pool of bad choices; it doesn't make the model smarter.
*   **Inconsistent Pool Sizes:** Unlike "Top-k" (which always limits the choice to exactly *k* words), Top-p is dynamic. If the model is very confident, the 90% pool might contain just 2 words. If the model is uncertain, the 90% pool might contain 500 words. This dynamic nature is usually a strength, but it means you can never guarantee exactly how many options the model is choosing from at any given step.

## Prompt vs Context vs Harness Engineering:

| Feature | Prompt Engineering | Context Engineering | Harness Engineering |
| :--- | :--- | :--- | :--- |
| **Definition** | Crafting the exact words, instructions, and examples sent to the model. | Selecting and injecting the right background information into the prompt. | Building the software architecture and pipelines that wrap around the model. |
| **Focus** | Instruction and tone. | Knowledge and grounding. | Orchestration and execution. |
| **Key Techniques** | Few-shot prompting, Chain-of-Thought, persona adoption, output formatting. | Retrieval-Augmented Generation (RAG), vector databases, chunking, semantic search. | API integrations, agent loops, guardrails, memory management, evaluation frameworks. |
| **Primary Goal** | Get the model to understand *how* to do the task and *what* format to output. | Give the model the exact *facts* or *data* it needs to complete the task accurately. | Build a reliable, scalable, and safe *system* or product around the LLM. |
| **Example** | "Act as a Python expert. Rewrite this code to be more efficient, and explain it to a 5-year-old." | "Based *only* on these three attached PDF manuals, how do I reset the router?" | A system that receives a user query, searches a database, calls a calculator tool, and filters out toxic outputs. |

#### Which is Better in What Case?

These three are not mutually exclusive; they actually stack on top of each other. However, you should focus on a specific one depending on the problem you are trying to solve:

*   **Focus on Prompt Engineering when:**
    *   You need to change the model's tone, style, or output format (e.g., "Output as a JSON object").
    *   The model has the knowledge, but struggles with the logic (use Chain-of-Thought).
    *   You are doing quick, one-off tasks in a chat interface like ChatGPT.
    *   *Best for: Formatting, logic routing, and behavioral adjustments.*

*   **Focus on Context Engineering when:**
    *   The model hallucinates or makes up facts.
    *   You need the model to answer questions about proprietary, private, or real-time data it wasn't trained on (e.g., your company's internal wiki).
    *   You are building a Q&A bot for a specific document, book, or codebase.
    *   *Best for: Accuracy, grounding, and domain-specific knowledge.*

*   **Focus on Harness Engineering when:**
    *   You are building a production-ready application for real users.
    *   The LLM needs to take actions in the real world (e.g., querying an API, sending an email, writing to a database).
    *   You need robust security, guardrails (preventing prompt injection), and error handling.
    *   The task requires an "Agent" that makes decisions in a multi-step loop.
    *   *Best for: Automation, scale, safety, and building autonomous agents.*